# 購買予測（オーディオブック）

## データ抽出

In [1]:
import numpy as np
from sklearn import preprocessing
import tensorflow as tf

raw_csv_data = np.loadtxt("Audiobooks_data.csv", delimiter=",")

unscaled_inputs_all = raw_csv_data[:,1:-1]
targets_all = raw_csv_data[:,-1]

## バランシング

In [2]:
num_one_targets = int(np.sum(targets_all))

zero_targets_counter = 0
indices_to_remove = []

for i in range(targets_all.shape[0]):
    if targets_all[i] ==0:
        zero_targets_counter += 1
        if zero_targets_counter > num_one_targets:
            indices_to_remove.append(i)

unscaled_inputs_equal_priors = np.delete(unscaled_inputs_all, indices_to_remove, axis=0)
targets_equal_priors = np.delete(targets_all, indices_to_remove, axis=0)

## 標準化とデータシャッフル

In [3]:
# 標準化
scaled_inputs = preprocessing.scale(unscaled_inputs_equal_priors)

# シャッフル
shuffled_indices = np.arange(scaled_inputs.shape[0])
np.random.shuffle(shuffled_indices)

shuffled_inputs = scaled_inputs[shuffled_indices]
shuffled_targets = targets_equal_priors[shuffled_indices]

## データセットの分割

In [4]:
samples_count = shuffled_inputs.shape[0]

# 80-10-10の割合で分割
train_samples_count = int(0.8 * samples_count)
validation_samples_count = int(0.1 * samples_count)
test_samples_count = samples_count - train_samples_count - validation_samples_count

# 訓練用データを
train_inputs = shuffled_inputs[:train_samples_count]
train_targets = shuffled_targets[:train_samples_count]

# 検証用データ
validation_inputs = shuffled_inputs[train_samples_count:train_samples_count+validation_samples_count]
validation_targets = shuffled_targets[train_samples_count:train_samples_count+validation_samples_count]

# テスト用データ
test_inputs = shuffled_inputs[train_samples_count+validation_samples_count:]
test_targets = shuffled_targets[train_samples_count+validation_samples_count:]


# 作成したデータ内の１と０の割合確認
print(np.sum(train_targets), train_samples_count, np.sum(train_targets) / train_samples_count)
print(np.sum(validation_targets), validation_samples_count, np.sum(validation_targets) / validation_samples_count)
print(np.sum(test_targets), test_samples_count, np.sum(test_targets) / test_samples_count)

1775.0 3579 0.49594858899133837
234.0 447 0.5234899328859061
228.0 448 0.5089285714285714


## .npz形式での保存

In [5]:
np.savez('Audiobooks_data_train', inputs=train_inputs, targets=train_targets)
np.savez('Audiobooks_data_validation', inputs=validation_inputs, targets=validation_targets)
np.savez('Audiobooks_data_test', inputs=test_inputs, targets=test_targets)

## データ

In [6]:
# 訓練用データの変数への代入
npz = np.load('Audiobooks_data_train.npz')
train_inputs = npz['inputs'].astype(np.float64)
train_targets = npz['targets'].astype(np.int64)


# 検証用データの変数への代入
npz = np.load('Audiobooks_data_validation.npz')
validation_inputs, validation_targets = npz['inputs'].astype(np.float64), npz['targets'].astype(np.int64)


# テストデータの変数への代入
npz = np.load('Audiobooks_data_test.npz')
test_inputs, test_targets = npz['inputs'].astype(np.float64), npz['targets'].astype(np.int64)

## モデル作成

In [18]:
# 入力数,出力の数の定義
input_size = 10
output_size = 2

# 隠れ層のユニット数の定義
hidden_layer_size = 50
    
# モデル作成
model = tf.keras.Sequential([
    tf.keras.layers.Dense(hidden_layer_size, activation='relu'), 
    tf.keras.layers.Dense(hidden_layer_size, activation='relu'), 
    tf.keras.layers.Dense(output_size, activation='softmax') 
])


# 損失関数と最適化アルゴリズムの定義
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])



# バッチサイズと、エポック数の設定
batch_size = 100
max_epochs = 100

# 過学習の対策
early_stopping = tf.keras.callbacks.EarlyStopping(patience=2)


# モデルへのデータ入力
model.fit(train_inputs, 
          train_targets, 
          batch_size=batch_size,
          epochs=max_epochs,
          callbacks=[early_stopping],
          validation_data=(validation_inputs, validation_targets),
          verbose = 2 
          )  

Epoch 1/100
36/36 - 1s - 36ms/step - accuracy: 0.8019 - loss: 0.4785 - val_accuracy: 0.8881 - val_loss: 0.3353
Epoch 2/100
36/36 - 0s - 10ms/step - accuracy: 0.8818 - loss: 0.3317 - val_accuracy: 0.8926 - val_loss: 0.3063
Epoch 3/100
36/36 - 0s - 10ms/step - accuracy: 0.8894 - loss: 0.3133 - val_accuracy: 0.9016 - val_loss: 0.2880
Epoch 4/100
36/36 - 0s - 10ms/step - accuracy: 0.8913 - loss: 0.3035 - val_accuracy: 0.9016 - val_loss: 0.2802
Epoch 5/100
36/36 - 0s - 9ms/step - accuracy: 0.8935 - loss: 0.2975 - val_accuracy: 0.9016 - val_loss: 0.2742
Epoch 6/100
36/36 - 0s - 9ms/step - accuracy: 0.8969 - loss: 0.2936 - val_accuracy: 0.8949 - val_loss: 0.2769
Epoch 7/100
36/36 - 0s - 9ms/step - accuracy: 0.8958 - loss: 0.2898 - val_accuracy: 0.8971 - val_loss: 0.2711
Epoch 8/100
36/36 - 0s - 9ms/step - accuracy: 0.8980 - loss: 0.2875 - val_accuracy: 0.9038 - val_loss: 0.2605
Epoch 9/100
36/36 - 0s - 9ms/step - accuracy: 0.8969 - loss: 0.2866 - val_accuracy: 0.9016 - val_loss: 0.2669
Epoch 

## モデルテスト

In [19]:
test_loss, test_accuracy = model.evaluate(test_inputs, test_targets)
print('\nTest loss: {0:.2f}. Test accuracy: {1:.2f}%'.format(test_loss, test_accuracy*100.))

14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8951 - loss: 0.2867 

Test loss: 0.29. Test accuracy: 89.51%
